# Демо проектных упражнений модуля 15 — Поиск программ и проверка правила остановки

**Демо-реализация, не готовое решение полномасштабного проекта.** Исполняемый генератор многочленов, проверка видимых примеров, отбор, ранняя остановка и скрытая оценка. Первый опыт сравнивает четыре политики, второй выбирает бюджет на отдельных калибровочных задачах. Все вызовы считаются; LLM и API не используются.
Все входы малы, детерминированы и работают без сети на CPU. Цель — проверить
интерфейсы, метрики, инварианты и структуру отчёта до расходования GPU/API-бюджета.

In [ ]:
from __future__ import annotations
import importlib.util, subprocess, sys
for package in ("numpy", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])


import math
import random
import time
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np

SEED = 20260801
random.seed(SEED)
np.random.seed(SEED)
print(f"воспроизводимый тестовый запуск; seed={SEED}")
DEMO_LANGUAGE = 'ru'


Задача — восстановить многочлен степени не выше двух. Генератору известно значение в нуле; два других коэффициента он выбирает случайно. Верификатор проверяет значения в −1 и 1. Сначала сравним одну попытку, лучший результат из 16 и две политики остановки. Для каждой задачи политики используют одну и ту же последовательность кандидатов. Скрытые входы 2 и 3 участвуют только в итоговой оценке.

In [ ]:
"""Generate, check, select and stop on a finite polynomial-program task.

No LLM or API is used. The generator samples two unknown coefficients; a visible
example fixes the constant term. Tests execute the candidates rather than drawing
precomputed correctness flags. This isolates selection and accounting.
"""
from dataclasses import dataclass, asdict
import itertools
import json
import random
import re

@dataclass(frozen=True)
class ProgramTask:
    task_id: int
    a: int
    b: int
    c: int
    def target(self, x):
        return self.a*x*x+self.b*x+self.c

TASKS = [ProgramTask(i, *abc) for i, abc in enumerate(itertools.product(range(-2, 3), repeat=3))]
CALIBRATION = [t for t in TASKS if t.c == 0]
HELD_OUT = [t for t in TASKS if t.c != 0]
assert len(CALIBRATION) == 25 and len(HELD_OUT) == 100
assert not {t.task_id for t in CALIBRATION} & {t.task_id for t in HELD_OUT}

def execute_program(source, x):
    # A deliberately restricted grammar; no eval or arbitrary code execution.
    match = re.fullmatch(r"(-?\d+)\*x\*x\+(-?\d+)\*x\+(-?\d+)", source)
    if match is None:
        raise ValueError("not a polynomial in the declared grammar")
    a, b, c = map(int, match.groups())
    return a*x*x+b*x+c

class ProposalGenerator:
    def __init__(self):
        self.calls = 0
    def propose(self, task_id, visible_f0, candidate_index):
        # Same deterministic random stream for each task/policy comparison.
        # Hidden coefficients, test outputs and future candidates are unavailable.
        rng = random.Random(150_000 + task_id*10_000 + candidate_index)
        a, b = rng.randrange(-2, 3), rng.randrange(-2, 3)
        self.calls += 1
        return f"{a}*x*x+{b}*x+{visible_f0}"

class ProgramVerifier:
    def __init__(self):
        self.calls = 0
        self.test_cases = 0
    def score(self, source, examples):
        self.calls += 1
        results = []
        for x, target in examples:
            results.append(execute_program(source, x) == target)
            self.test_cases += 1
        return sum(results)/len(results), results

def search_program(task, *, budget, threshold=None):
    if isinstance(budget, bool) or not isinstance(budget, int) or budget <= 0:
        raise ValueError("budget must be a positive integer")
    if threshold is not None and threshold not in (.5, 1.):
        raise ValueError("threshold must be .5, 1, or None")
    generator, verifier = ProposalGenerator(), ProgramVerifier()
    # The generator sees only f(0). The verifier receives two more examples.
    examples = [(x, task.target(x)) for x in (1, -1)]
    records = []
    for index in range(budget):
        source = generator.propose(task.task_id, task.target(0), index)
        score, checks = verifier.score(source, examples)
        records.append(dict(index=index, source=source, score=score, checks=checks))
        if threshold is not None and score >= threshold:
            break
    # Deterministic earliest candidate on a score tie. No extra generation or scoring.
    best = max(range(len(records)), key=lambda i: records[i]['score'])
    assert generator.calls == verifier.calls == len(records)
    assert verifier.test_cases == 2*len(records)
    if threshold is not None and len(records) < budget:
        assert records[-1]['score'] >= threshold
        assert all(r['score'] < threshold for r in records[:-1])
    # Hidden tests are evaluation only; they cannot influence selection or stopping.
    before = (generator.calls, verifier.calls, verifier.test_cases)
    hidden = [all(execute_program(r['source'], x) == task.target(x) for x in (2, 3))
              for r in records]
    assert before == (generator.calls, verifier.calls, verifier.test_cases)
    result = dict(task_id=task.task_id, records=records, selected_index=best,
                  correct=hidden[best], observed_coverage=any(hidden),
                  generated=generator.calls, verifier_calls=verifier.calls,
                  visible_test_cases=verifier.test_cases, hidden_tests_max=2*len(records))
    assert json.loads(json.dumps(result)) == result
    return result

def evaluate_program_policy(tasks, *, budget, threshold=None):
    runs = [search_program(t, budget=budget, threshold=threshold) for t in tasks]
    n = len(runs)
    assert n > 0
    return dict(accuracy=sum(r['correct'] for r in runs)/n,
                mean_candidates=sum(r['generated'] for r in runs)/n,
                observed_coverage=sum(r['observed_coverage'] for r in runs)/n,
                generator_calls=sum(r['generated'] for r in runs),
                verifier_calls=sum(r['verifier_calls'] for r in runs),
                visible_test_cases=sum(r['visible_test_cases'] for r in runs)), runs

def run_program_search():
    results = {}
    traces = {}
    for name, budget, threshold in [('one', 1, None), ('fixed16', 16, None),
                                    ('stop_partial', 16, .5), ('stop_complete', 16, 1.)]:
        metrics, records = evaluate_program_policy(HELD_OUT, budget=budget, threshold=threshold)
        results[name], traces[name] = metrics, records
    assert results['fixed16']['generator_calls'] == 1600
    assert results['one']['generator_calls'] == 100
    assert results['stop_complete']['accuracy'] == results['fixed16']['accuracy']
    assert results['stop_complete']['mean_candidates'] < 16
    assert results['stop_partial']['accuracy'] < results['stop_complete']['accuracy']
    # Passing all three x values (one visible to the generator, two to the verifier)
    # identifies a degree-at-most-two polynomial exactly. This guarantee is grammar-specific.
    for name in ('fixed16', 'stop_complete'):
        for record in traces[name]:
            assert (record['records'][record['selected_index']]['score'] == 1) == record['correct']
    for name, result in results.items(): print(name, result)
    return results, traces


program_results, program_traces = run_program_search()

Полная проверка сохраняет точность 0.45 лучшего из 16, сокращая вызовы с 1600 до 1225. Остановка после одного успешного теста требует 440 вызовов, но даёт точность лишь 0.15. Здесь совпадение в трёх точках определяет многочлен однозначно; такое объяснение нельзя переносить на произвольную программу. Журнал позволяет найти первый момент остановки и проверить, что последующие кандидаты действительно не генерировались.

Теперь выберем бюджет по отдельным 25 задачам. До расчёта зафиксированы порог проверки 1, набор бюджетов и целевая эмпирическая точность 0.60. Среди подходящих вариантов выбирается тот, который потребовал меньше вызовов. После этого политика замораживается и применяется к 100 отложенным задачам; их ответы не участвуют в выборе бюджета.

In [ ]:
# Choose the cheapest candidate budget meeting a declared calibration target.
# The target is empirical accuracy, not a statistical lower confidence bound.
target_accuracy = .60
calibration_rows = []
for budget in (1, 4, 8, 16, 32):
    metrics, _ = evaluate_program_policy(CALIBRATION, budget=budget, threshold=1.)
    calibration_rows.append(dict(budget=budget, **metrics))
eligible = [r for r in calibration_rows if r['accuracy'] >= target_accuracy]
if not eligible:
    raise RuntimeError("No calibration budget meets the target")
chosen = min(eligible, key=lambda r: (r['mean_candidates'], r['budget']))
frozen_policy = dict(budget=chosen['budget'], threshold=1.)
held_out, held_out_runs = evaluate_program_policy(HELD_OUT, **frozen_policy)
assert chosen['budget'] == 32
assert held_out['generator_calls'] == held_out['verifier_calls']
assert held_out['visible_test_cases'] == 2*held_out['verifier_calls']
artifact = dict(policy=frozen_policy, calibration=calibration_rows, held_out=held_out)
assert json.loads(json.dumps(artifact)) == artifact
print("calibration:", calibration_rows)
print("frozen policy:", frozen_policy, "held-out:", held_out)
# Inspect an actual wrong selected program; never use this inspection to retune test results.
failure = next((r for r in held_out_runs if not r['correct']), None)
print("held-out failure:", failure)

На фиксированном наборе выбран лимит 32. Сравните калибровочную и отложенную точность, но не подстраивайте политику по показанному отложенному провалу. Порог 0.60 относится к наблюдаемой доле успехов на 25 задачах: это не доверительная граница и не обещание результата на другой выборке. Сохранённый JSON связывает итоговую метрику с конкретной политикой и её затратами.

## Переход к полному проекту

Калибровочная доля 0.60 — эмпирический порог на 25 задачах, а не гарантия будущего качества. Для настоящего поиска кода разделяйте семейства задач, ограничивайте среду исполнения и учитывайте время проверки и все сгенерированные токены. Сравнивайте при одинаковом бюджете и не раскрывайте скрытые тесты генератору. Полнота трёх точек верна только для данной грамматики.

### Минимальный отчёт

1. Версии данных, модели, кода и seed.
2. Основная метрика, а также качество, стоимость, задержка и память там, где это применимо.
3. Базовая линия, одинаковый бюджет и хотя бы один отрицательный контроль.
4. Неудачные прогоны и границы обоснованных выводов.